# Model Experimentation & Comparison Guide

This notebook demonstrates how to run crop classification using different model architectures (ModelType) available in the CropClassifier package.

To save time and computational resources, we recommend downloading satellite data and building features once, then running inference with different models on the same processed dataset.

## 1. Setup and Initialization

Import the required modules and define common configuration settings for your study area and target period.

In [ ]:
from pathlib import Path
from crop_classifier import CropClassifier, ModelType, OutputFormat

# Define global configuration
SHAPEFILE_PATH = "../data/raw/fields.shp"
GEE_PROJECT = "your-gee-project-id"
YEAR = 2015
OUTPUT_DIR = "../data/model_experiments"

# Initialize classifier with default parameters
classifier = CropClassifier(
    shapefile_path=SHAPEFILE_PATH,
    gee_project=GEE_PROJECT,
    year=YEAR,
    output_dir=OUTPUT_DIR,
    use_zonal_spectral=True,
    output_format=OutputFormat.TABLE,
    epsg_code="EPSG:32637"
)

## 2. Prepare Data (Run Once)

Download Google Earth Engine imagery and generate features once. This dataset will serve as the common baseline for evaluating all models.

In [ ]:
# Step 1: Download raw satellite and meteorological data
print("Downloading GEE data...")
classifier.download_data()

# Step 2: Generate features and obtain the feature file path
print("Generating predictor features...")
processed_features_path = classifier.build_features()

print(f"Features ready at: {processed_features_path}")

## 3. Iterating Over Multiple Models

Loop through different model types (e.g., FINETUNED, BASE, or custom models) and execute predictions on the pre-built features. Each model output will be saved with a distinct file prefix.

In [ ]:
# Define model types you want to evaluate
models_to_evaluate = [
    ModelType.FINETUNED,
    ModelType.BASE,
]

# Run inference for each model architecture
for model_type in models_to_evaluate:
    print(f"\n--- Running Inference with Model: {model_type.value} ---")
    
    # Update current model type
    classifier.model_type = model_type
    
    # Define custom output prefix for this model
    custom_output_prefix = classifier.output_dir / "final" / f"{classifier.input_file.stem}_{model_type.value}"
    
    # Execute prediction using pre-built features
    classifier.predict(
        processed_path=processed_features_path,
        output_prefix=custom_output_prefix
    )

## 4. Comparing Model Outputs

After generating predictions from different models, load the resulting output tables to evaluate performance metrics, classification agreements, or confidence scores.

In [ ]:
import pandas as pd

# Load prediction outputs
finetuned_results = pd.read_csv(f"{OUTPUT_DIR}/final/fields_finetuned.csv")
base_results = pd.read_csv(f"{OUTPUT_DIR}/final/fields_base.csv")

# Compare predicted crop classes across models
comparison_df = pd.DataFrame({
    "field_id": finetuned_results["id"],
    "finetuned_pred": finetuned_results["crop_label"],
    "base_pred": base_results["crop_label"],
})

# Calculate classification agreement rate
agreement_rate = (comparison_df["finetuned_pred"] == comparison_df["base_pred"]).mean()
print(f"Model Agreement Rate: {agreement_rate:.2%}")

comparison_df.head()